# 05 — Negative Binomial Regression

This notebook fits the manuscript's primary Negative Binomial regression model using the same predictors and population offset as the Week 3 Poisson baseline.

Model form:

Dengue Cases ~ Negative Binomial(mu)

log(mu) = intercept + b1 × Population Density + year terms + log(Population)

The population term is included as a log-offset to model dengue case counts relative to the population exposed to risk.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path.cwd().parent))

In [3]:
panel_path = Path("../data/04_validated/lgu_year_panel.csv")

panel = pd.read_csv(panel_path)

print("Shape:", panel.shape)
print("\nColumns:")
print(panel.columns.tolist())

panel.head()

Shape: (85, 8)

Columns:
['LGU', 'Year', 'Dengue Cases', 'Population', 'Land Area', 'Population Density', 'Incidence Rate', 'Status']


,LGU,Year,Dengue Cases,Population,Land Area,Population Density,Incidence Rate,Status
0,Caloocan,2021,1051,1674424.25,55.8,30007.603047,62.767844,interpolated
1,Caloocan,2022,4371,1687264.50,55.8,30237.715054,259.058375,interpolated
2,Caloocan,2023,2334,1700104.75,55.8,30467.827061,137.285658,interpolated
3,Caloocan,2024,3262,1712945.00,55.8,30697.939068,190.432267,official
4,Caloocan,2025,5451,1725785.25,55.8,30928.051075,315.856217,extrapolated


In [4]:
import statsmodels.api as sm

# Create year dummy variables, using 2021 as the reference year
year_dummies = pd.get_dummies(
    panel["Year"].astype(str),
    prefix="Year",
    drop_first=True
)

# Main predictor + year controls
X = pd.concat(
    [
        panel[["Population Density"]],
        year_dummies
    ],
    axis=1
)

# Add intercept
X = sm.add_constant(X)

# Dependent variable
y = panel["Dengue Cases"]

# Population exposure offset
offset = np.log(panel["Population"])

print("Design matrix shape:", X.shape)
print("\nPredictors:")
print(X.columns.tolist())

X = X.astype(float)
y = y.astype(float)
offset = offset.astype(float)

Design matrix shape: (85, 6)

Predictors:
['const', 'Population Density', 'Year_2022', 'Year_2023', 'Year_2024', 'Year_2025']


In [5]:
nb_model = sm.GLM(
    y,
    X,
    family=sm.families.NegativeBinomial(),
    offset=offset
)

nb_results = nb_model.fit()

print(nb_results.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           Dengue Cases   No. Observations:                   85
Model:                            GLM   Df Residuals:                       79
Model Family:        NegativeBinomial   Df Model:                            5
Link Function:                    Log   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -688.38
Date:                Fri, 25 Sep 2026   Deviance:                       10.911
Time:                        14:27:37   Pearson chi2:                     11.7
No. Iterations:                     8   Pseudo R-squ. (CS):             0.2077
Covariance Type:            nonrobust                                         
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                 -6.9047      0

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/genmod/families/family.py:1367: ValueWarning: Negative binomial dispersion parameter alpha not set. Using default value alpha=1.0.
  warnings.warn("Negative binomial dispersion parameter alpha not "


In [6]:
# Fit Negative Binomial regression with estimated dispersion (NB2)
nb_model = sm.NegativeBinomial(
    y,
    X,
    offset=offset,
    loglike_method="nb2"
)

nb_results = nb_model.fit(
    disp=False,
    maxiter=1000
)

print(nb_results.summary())

                     NegativeBinomial Regression Results                      
Dep. Variable:           Dengue Cases   No. Observations:                   85
Model:               NegativeBinomial   Df Residuals:                       79
Method:                           MLE   Df Model:                            5
Date:                Fri, 25 Sep 2026   Pseudo R-squ.:                     nan
Time:                        14:27:37   Log-Likelihood:                    nan
converged:                      False   LL-Null:                       -677.70
Covariance Type:            nonrobust   LLR p-value:                       nan
                         coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                -41.0275        nan        nan        nan         nan         nan
Population Density     0.0010        nan        nan        nan         nan         nan
Year_2022           

/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:3384: RuntimeWarning: overflow encountered in exp
  alpha = np.exp(params[-1])
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:3379: RuntimeWarning: divide by zero encountered in log
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:3379: RuntimeWarning: invalid value encountered in multiply
  llf = coeff + size*np.log(prob) + endog*np.log(1-prob)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:3448: RuntimeWarning: overflow encountered in exp
  alpha = np.exp(params[-1])
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:3472: RuntimeWarning: divide by zero encountered in log
  dalpha = (dgpart + np.log(a1)
/opt/anaconda3/lib/python3.13/site-packages/statsmodels/discrete/discrete_model.py:3472: RuntimeWarning: inva

In [7]:
# Rescale population density for numerical stability
panel["Density_per_1000"] = panel["Population Density"] / 1000

year_dummies = pd.get_dummies(
    panel["Year"].astype(str),
    prefix="Year",
    drop_first=True
)

X_nb = pd.concat(
    [
        panel[["Density_per_1000"]],
        year_dummies
    ],
    axis=1
)

X_nb = sm.add_constant(X_nb).astype(float)

y_nb = panel["Dengue Cases"].astype(float)
offset_nb = np.log(panel["Population"]).astype(float)

print(X_nb.describe())

       const  Density_per_1000  Year_2022  Year_2023  Year_2024  Year_2025
count   85.0         85.000000  85.000000  85.000000  85.000000  85.000000
mean     1.0         25.490830   0.200000   0.200000   0.200000   0.200000
std      0.0         15.718034   0.402374   0.402374   0.402374   0.402374
min      1.0          6.322115   0.000000   0.000000   0.000000   0.000000
25%      1.0         15.476856   0.000000   0.000000   0.000000   0.000000
50%      1.0         21.775290   0.000000   0.000000   0.000000   0.000000
75%      1.0         28.933532   0.000000   0.000000   0.000000   0.000000
max      1.0         76.725751   1.000000   1.000000   1.000000   1.000000


In [8]:
nb_model = sm.NegativeBinomial(
    y_nb,
    X_nb,
    offset=offset_nb,
    loglike_method="nb2"
)

nb_results = nb_model.fit(
    method="bfgs",
    maxiter=1000,
    disp=False
)

print(nb_results.summary())

                     NegativeBinomial Regression Results                      
Dep. Variable:           Dengue Cases   No. Observations:                   85
Model:               NegativeBinomial   Df Residuals:                       79
Method:                           MLE   Df Model:                            5
Date:                Fri, 25 Sep 2026   Pseudo R-squ.:                 0.06728
Time:                        14:27:37   Log-Likelihood:                -632.11
converged:                       True   LL-Null:                       -677.70
Covariance Type:            nonrobust   LLR p-value:                 3.786e-18
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -6.9086      0.104    -66.662      0.000      -7.112      -6.705
Density_per_1000    -0.0065      0.002     -2.896      0.004      -0.011      -0.002
Year_2022            1.4221 

In [9]:
# Refit the same Negative Binomial model with
# cluster-robust standard errors at the LGU level

nb_results_clustered = nb_model.fit(
    method="bfgs",
    maxiter=1000,
    disp=False,
    cov_type="cluster",
    cov_kwds={
        "groups": panel["LGU"]
    }
)

print(nb_results_clustered.summary())

                     NegativeBinomial Regression Results                      
Dep. Variable:           Dengue Cases   No. Observations:                   85
Model:               NegativeBinomial   Df Residuals:                       79
Method:                           MLE   Df Model:                            5
Date:                Fri, 25 Sep 2026   Pseudo R-squ.:                 0.06728
Time:                        14:27:37   Log-Likelihood:                -632.11
converged:                       True   LL-Null:                       -677.70
Covariance Type:              cluster   LLR p-value:                 3.786e-18
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -6.9086      0.152    -45.574      0.000      -7.206      -6.611
Density_per_1000    -0.0065      0.004     -1.516      0.130      -0.015       0.002
Year_2022            1.4221 

In [10]:
# Build a reporting table for the clustered Negative Binomial model

params = nb_results_clustered.params
se = nb_results_clustered.bse
pvalues = nb_results_clustered.pvalues

report_table = pd.DataFrame({
    "Coefficient": params,
    "Direction": np.where(params > 0, "Positive", "Negative"),
    "Std_Error": se,
    "P_Value": pvalues,
    "IRR": np.exp(params)
})

# Alpha is the dispersion parameter and not a regression predictor
report_table.loc["alpha", "IRR"] = np.nan
report_table.loc["alpha", "Direction"] = "Dispersion parameter"

# Format p-values for reporting purposes
def format_p_value(p):
    if p < 0.001:
        return "< .001"
    return f"{p:.4f}"

report_table["P_Value"] = report_table["P_Value"].apply(format_p_value)

# Round numeric columns
report_table["Coefficient"] = report_table["Coefficient"].round(4)
report_table["Std_Error"] = report_table["Std_Error"].round(4)
report_table["IRR"] = report_table["IRR"].round(4)

report_table

,Coefficient,Direction,Std_Error,P_Value,IRR
const,-6.9086,Negative,0.1516,< .001,0.0010
Density_per_1000,-0.0065,Negative,0.0043,0.1296,0.9935
Year_2022,1.4221,Positive,0.0561,< .001,4.1458
Year_2023,0.7725,Positive,0.0870,< .001,2.1651
Year_2024,1.2211,Positive,0.1142,< .001,3.3909
Year_2025,1.2711,Positive,0.1172,< .001,3.5648
alpha,0.1240,Dispersion parameter,0.0267,< .001,NaN


In [11]:
# Model fit statistics:
# McFadden's pseudo-R² and Negative Binomial deviance

# McFadden's pseudo-R²
mcfadden_r2 = 1 - (nb_results_clustered.llf / nb_results_clustered.llnull)

# Estimated dispersion parameter
alpha_hat = nb_results_clustered.params["alpha"]

# Regression coefficients only (exclude alpha)
beta_hat = nb_results_clustered.params.drop("alpha")

# Fitted mean counts
mu_hat = np.exp(
    np.dot(X_nb, beta_hat) + offset_nb
)

# Compute Negative Binomial deviance using the estimated alpha
nb_family = sm.families.NegativeBinomial(alpha=alpha_hat)

model_deviance = nb_family.deviance(
    y_nb,
    mu_hat
)

print(f"Log-Likelihood: {nb_results_clustered.llf:.4f}")
print(f"Null Log-Likelihood: {nb_results_clustered.llnull:.4f}")
print(f"McFadden's Pseudo-R²: {mcfadden_r2:.4f}")
print(f"Negative Binomial Deviance: {model_deviance:.4f}")
print(f"Estimated alpha: {alpha_hat:.4f}")

Log-Likelihood: -632.1121
Null Log-Likelihood: -677.7048
McFadden's Pseudo-R²: 0.0673
Negative Binomial Deviance: 86.9229
Estimated alpha: 0.1240


In [12]:
# Deviance residual diagnostics for the fitted Negative Binomial model

deviance_residuals = nb_family.resid_dev(
    y_nb,
    mu_hat
)

residual_summary = pd.Series(
    deviance_residuals,
    name="Deviance Residual"
).describe()

print(residual_summary)

count    85.000000
mean     -0.117816
std       1.010322
min      -2.511196
25%      -0.736847
50%      -0.255242
75%       0.446196
max       2.638997
Name: Deviance Residual, dtype: float64


In [13]:
# Identify observations with relatively large deviance residuals

residual_check = panel[["LGU", "Year", "Dengue Cases"]].copy()
residual_check["Predicted Cases"] = mu_hat
residual_check["Deviance Residual"] = deviance_residuals
residual_check["Abs Deviance Residual"] = np.abs(deviance_residuals)

residual_check.sort_values(
    "Abs Deviance Residual",
    ascending=False
).head(10)

,LGU,Year,Dengue Cases,Predicted Cases,Deviance Residual,Abs Deviance Residual
63,Pateros,2024,494,218.620600,2.638997,2.638997
37,Muntinlupa,2023,383,1086.854369,-2.511196,2.511196
29,Manila,2025,8601,4131.001048,2.369504,2.369504
35,Muntinlupa,2021,198,498.337536,-2.247090,2.247090
42,Navotas,2023,191,452.626156,-2.116975,2.116975
16,Malabon,2022,2520,1359.095230,1.949100,1.949100
17,Malabon,2023,1299,713.410213,1.881290,1.881290
62,Pateros,2023,249,138.553629,1.801027,1.801027
51,Pasay,2022,746,1501.496054,-1.773193,1.773193
50,Pasay,2021,186,360.161499,-1.666997,1.666997


In [14]:
# Influence diagnostics using a Negative Binomial GLM
# with alpha fixed at the value estimated by the primary NB2 model

nb_glm_influence_model = sm.GLM(
    y_nb,
    X_nb,
    family=sm.families.NegativeBinomial(alpha=alpha_hat),
    offset=offset_nb
)

nb_glm_influence_results = nb_glm_influence_model.fit()

influence = nb_glm_influence_results.get_influence(observed=True)

cooks_d = influence.cooks_distance[0]

influence_check = panel[
    ["LGU", "Year", "Dengue Cases"]
].copy()

influence_check["Predicted Cases"] = mu_hat
influence_check["Cooks Distance"] = cooks_d

# Common screening threshold
cook_threshold = 4 / len(panel)

print(f"Cook's distance screening threshold (4/n): {cook_threshold:.4f}")

influence_check.sort_values(
    "Cooks Distance",
    ascending=False
).head(10)

Cook's distance screening threshold (4/n): 0.0471


,LGU,Year,Dengue Cases,Predicted Cases,Cooks Distance
29,Manila,2025,8601,4131.001048,1.094801
63,Pateros,2024,494,218.620600,0.461561
16,Malabon,2022,2520,1359.095230,0.133715
62,Pateros,2023,249,138.553629,0.127443
17,Malabon,2023,1299,713.410213,0.119838
28,Manila,2024,5511,3915.056133,0.087518
80,Valenzuela,2021,1088,648.777661,0.080962
81,Valenzuela,2022,4504,2698.307659,0.077154
61,Pateros,2022,412,263.314779,0.056257
12,Makati,2023,1136,749.066056,0.039535


In [15]:
import warnings

# Leave-one-LGU-out validation
# Use the full-model estimates as starting values to improve convergence.

loo_results = []

# Starting parameters from the successfully converged full NB2 model
start_params = nb_results.params.values

for omitted_lgu in sorted(panel["LGU"].unique()):

    subset = panel[panel["LGU"] != omitted_lgu].copy()

    subset["Density_per_1000"] = (
        subset["Population Density"] / 1000
    )

    subset_year_dummies = pd.get_dummies(
        subset["Year"].astype(str),
        prefix="Year",
        drop_first=True
    )

    X_loo = pd.concat(
        [
            subset[["Density_per_1000"]],
            subset_year_dummies
        ],
        axis=1
    )

    X_loo = sm.add_constant(X_loo).astype(float)

    y_loo = subset["Dengue Cases"].astype(float)
    offset_loo = np.log(subset["Population"]).astype(float)

    loo_model = sm.NegativeBinomial(
        y_loo,
        X_loo,
        offset=offset_loo,
        loglike_method="nb2"
    )

    try:
        # Suppress expected numerical RuntimeWarnings during
        # the optimizer's temporary trial steps.
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)

            loo_fit = loo_model.fit(
                start_params=start_params,
                method="bfgs",
                maxiter=2000,
                disp=False
            )

        density_coef = loo_fit.params["Density_per_1000"]

        loo_results.append({
            "Omitted_LGU": omitted_lgu,
            "Converged": loo_fit.mle_retvals["converged"],
            "Density_Coefficient": density_coef,
            "Density_IRR": np.exp(density_coef),
            "Alpha": loo_fit.params["alpha"],
            "Log_Likelihood": loo_fit.llf
        })

    except Exception:
        loo_results.append({
            "Omitted_LGU": omitted_lgu,
            "Converged": False,
            "Density_Coefficient": np.nan,
            "Density_IRR": np.nan,
            "Alpha": np.nan,
            "Log_Likelihood": np.nan
        })

loo_table = pd.DataFrame(loo_results)

loo_table.round({
    "Density_Coefficient": 4,
    "Density_IRR": 4,
    "Alpha": 4,
    "Log_Likelihood": 2
})

,Omitted_LGU,Converged,Density_Coefficient,Density_IRR,Alpha,Log_Likelihood
0,Caloocan,True,-0.0063,0.9937,0.1276,-591.49
1,Las Piñas,True,-0.0067,0.9933,0.1294,-596.39
2,Makati,True,-0.0060,0.9941,0.1265,-596.23
3,Malabon,True,-0.0065,0.9935,0.1175,-592.59
4,Mandaluyong,True,-0.0054,0.9946,0.1277,-598.73
5,Manila,True,-0.0175,0.9827,0.1092,-583.88
6,Marikina,True,-0.0067,0.9933,0.1294,-597.96
7,Muntinlupa,True,-0.0078,0.9922,0.1038,-590.61
8,Navotas,True,-0.0065,0.9936,0.1230,-599.64
9,Parañaque,True,-0.0070,0.9931,0.1290,-595.74


In [16]:
# Sensitivity analysis excluding 2025

panel_no_2025 = panel[panel["Year"] != 2025].copy()

panel_no_2025["Density_per_1000"] = (
    panel_no_2025["Population Density"] / 1000
)

year_dummies_no_2025 = pd.get_dummies(
    panel_no_2025["Year"].astype(str),
    prefix="Year",
    drop_first=True
)

X_no_2025 = pd.concat(
    [
        panel_no_2025[["Density_per_1000"]],
        year_dummies_no_2025
    ],
    axis=1
)

X_no_2025 = sm.add_constant(X_no_2025).astype(float)

y_no_2025 = panel_no_2025["Dengue Cases"].astype(float)
offset_no_2025 = np.log(
    panel_no_2025["Population"]
).astype(float)

nb_model_no_2025 = sm.NegativeBinomial(
    y_no_2025,
    X_no_2025,
    offset=offset_no_2025,
    loglike_method="nb2"
)

nb_results_no_2025 = nb_model_no_2025.fit(
    method="bfgs",
    maxiter=2000,
    disp=False
)

print(nb_results_no_2025.summary())

                     NegativeBinomial Regression Results                      
Dep. Variable:           Dengue Cases   No. Observations:                   68
Model:               NegativeBinomial   Df Residuals:                       63
Method:                           MLE   Df Model:                            4
Date:                Fri, 25 Sep 2026   Pseudo R-squ.:                 0.07509
Time:                        14:27:37   Log-Likelihood:                -500.77
converged:                       True   LL-Null:                       -541.43
Covariance Type:            nonrobust   LLR p-value:                 9.194e-17
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -6.8419      0.110    -62.052      0.000      -7.058      -6.626
Density_per_1000    -0.0092      0.003     -3.524      0.000      -0.014      -0.004
Year_2022            1.4183 

In [17]:
# Sensitivity analysis excluding 2025

panel_no_2025 = panel[panel["Year"] != 2025].copy()

panel_no_2025["Density_per_1000"] = (
    panel_no_2025["Population Density"] / 1000
)

year_dummies_no_2025 = pd.get_dummies(
    panel_no_2025["Year"].astype(str),
    prefix="Year",
    drop_first=True
)

X_no_2025 = pd.concat(
    [
        panel_no_2025[["Density_per_1000"]],
        year_dummies_no_2025
    ],
    axis=1
)

X_no_2025 = sm.add_constant(X_no_2025).astype(float)

y_no_2025 = panel_no_2025["Dengue Cases"].astype(float)

offset_no_2025 = np.log(
    panel_no_2025["Population"]
).astype(float)

# Same Negative Binomial NB2 specification
nb_model_no_2025 = sm.NegativeBinomial(
    y_no_2025,
    X_no_2025,
    offset=offset_no_2025,
    loglike_method="nb2"
)

# Same LGU-clustered covariance approach as the primary model
nb_results_no_2025 = nb_model_no_2025.fit(
    method="bfgs",
    maxiter=2000,
    disp=False,
    cov_type="cluster",
    cov_kwds={
        "groups": panel_no_2025["LGU"]
    }
)

print(nb_results_no_2025.summary())

                     NegativeBinomial Regression Results                      
Dep. Variable:           Dengue Cases   No. Observations:                   68
Model:               NegativeBinomial   Df Residuals:                       63
Method:                           MLE   Df Model:                            4
Date:                Fri, 25 Sep 2026   Pseudo R-squ.:                 0.07509
Time:                        14:27:37   Log-Likelihood:                -500.77
converged:                       True   LL-Null:                       -541.43
Covariance Type:              cluster   LLR p-value:                 9.194e-17
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -6.8419      0.153    -44.745      0.000      -7.142      -6.542
Density_per_1000    -0.0092      0.004     -2.118      0.034      -0.018      -0.001
Year_2022            1.4183 

In [18]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Random Forest feature set
rf_data = panel.copy()

rf_data["Density_per_1000"] = (
    rf_data["Population Density"] / 1000
)

# Use the same substantive predictors:
# population density + year
rf_year_dummies = pd.get_dummies(
    rf_data["Year"].astype(str),
    prefix="Year",
    drop_first=True
)

X_rf = pd.concat(
    [
        rf_data[["Density_per_1000", "Population"]],
        rf_year_dummies
    ],
    axis=1
).astype(float)

y_rf = rf_data["Dengue Cases"].astype(float)

# Keep observations from the same LGU together during validation
groups = rf_data["LGU"]

print("Random Forest design matrix shape:", X_rf.shape)
print("\nFeatures:")
print(X_rf.columns.tolist())

Random Forest design matrix shape: (85, 6)

Features:
['Density_per_1000', 'Population', 'Year_2022', 'Year_2023', 'Year_2024', 'Year_2025']


In [19]:
# Sensitivity analysis excluding Makati and Taguig
# Tests whether the administrative boundary issue materially affects
# the primary Negative Binomial regression results.

boundary_affected_lgus = ["Makati", "Taguig"]

panel_no_boundary_lgus = panel[
    ~panel["LGU"].isin(boundary_affected_lgus)
].copy()

print("Primary model observations:", len(panel))
print("Sensitivity model observations:", len(panel_no_boundary_lgus))
print("LGUs in primary model:", panel["LGU"].nunique())
print(
    "LGUs in sensitivity model:",
    panel_no_boundary_lgus["LGU"].nunique()
)

Primary model observations: 85
Sensitivity model observations: 75
LGUs in primary model: 17
LGUs in sensitivity model: 15


In [28]:
# Recreate the primary NBR specification using the sensitivity dataset

panel_no_boundary_lgus["Density_per_1000"] = (
    panel_no_boundary_lgus["Population Density"] / 1000
)

year_dummies_boundary = pd.get_dummies(
    panel_no_boundary_lgus["Year"].astype(str),
    prefix="Year",
    drop_first=True
)

X_boundary = pd.concat(
    [
        panel_no_boundary_lgus[["Density_per_1000"]],
        year_dummies_boundary
    ],
    axis=1
)

X_boundary = sm.add_constant(X_boundary).astype(float)

y_boundary = panel_no_boundary_lgus["Dengue Cases"].astype(float)

offset_boundary = np.log(
    panel_no_boundary_lgus["Population"]
).astype(float)

print("Sensitivity design matrix shape:", X_boundary.shape)
print("Predictors:", X_boundary.columns.tolist())

Sensitivity design matrix shape: (75, 6)
Predictors: ['const', 'Density_per_1000', 'Year_2022', 'Year_2023', 'Year_2024', 'Year_2025']


In [29]:
# Fit Negative Binomial model excluding Makati and Taguig

nb_model_boundary = sm.NegativeBinomial(
    y_boundary,
    X_boundary,
    offset=offset_boundary,
    loglike_method="nb2"
)

nb_results_boundary = nb_model_boundary.fit(
    method="bfgs",
    maxiter=2000,
    disp=False,
    cov_type="cluster",
    cov_kwds={
        "groups": panel_no_boundary_lgus["LGU"]
    }
)

print(nb_results_boundary.summary())

                     NegativeBinomial Regression Results                      
Dep. Variable:           Dengue Cases   No. Observations:                   75
Model:               NegativeBinomial   Df Residuals:                       69
Method:                           MLE   Df Model:                            5
Date:                Fri, 25 Sep 2026   Pseudo R-squ.:                 0.06730
Time:                        14:29:44   Log-Likelihood:                -555.10
converged:                       True   LL-Null:                       -595.15
Covariance Type:              cluster   LLR p-value:                 7.969e-16
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -6.9594      0.164    -42.373      0.000      -7.281      -6.637
Density_per_1000    -0.0059      0.004     -1.328      0.184      -0.015       0.003
Year_2022            1.4235 

In [30]:
# Compare primary NBR with Makati–Taguig exclusion sensitivity model

sensitivity_comparison = pd.DataFrame({
    "Primary_Model": nb_results_clustered.params,
    "Exclude_Makati_Taguig": nb_results_boundary.params,
    "Primary_P_Value": nb_results_clustered.pvalues,
    "Sensitivity_P_Value": nb_results_boundary.pvalues
})

sensitivity_comparison["Coefficient_Change"] = (
    sensitivity_comparison["Exclude_Makati_Taguig"]
    - sensitivity_comparison["Primary_Model"]
)

sensitivity_comparison.round(4)

,Primary_Model,Exclude_Makati_Taguig,Primary_P_Value,Sensitivity_P_Value,Coefficient_Change
const,-6.9086,-6.9594,0.0000,0.0000,-0.0508
Density_per_1000,-0.0065,-0.0059,0.1296,0.1843,0.0007
Year_2022,1.4221,1.4235,0.0000,0.0000,0.0014
Year_2023,0.7725,0.7550,0.0000,0.0000,-0.0175
Year_2024,1.2211,1.2642,0.0000,0.0000,0.0431
Year_2025,1.2711,1.2911,0.0000,0.0000,0.0200
alpha,0.1240,0.1278,0.0000,0.0000,0.0038


In [31]:
# Structural-risk ranking sensitivity:
# Recalculate modeled risk for the 15 unaffected LGUs
# using the Makati–Taguig exclusion model.

beta_boundary = nb_results_boundary.params.drop("alpha")

mu_boundary = np.exp(
    np.dot(X_boundary, beta_boundary)
    + offset_boundary
)

structural_risk_boundary = panel_no_boundary_lgus[
    ["LGU", "Year", "Population", "Population Density"]
].copy()

structural_risk_boundary["Predicted_Cases"] = mu_boundary

structural_risk_boundary["Predicted_Incidence_per_100k"] = (
    structural_risk_boundary["Predicted_Cases"]
    / structural_risk_boundary["Population"]
) * 100000

lgu_risk_ranking_boundary = (
    structural_risk_boundary
    .groupby("LGU", as_index=False)
    .agg(
        Structural_Risk_Score=(
            "Predicted_Incidence_per_100k",
            "mean"
        )
    )
    .sort_values(
        "Structural_Risk_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

lgu_risk_ranking_boundary["Sensitivity_Rank"] = (
    np.arange(1, len(lgu_risk_ranking_boundary) + 1)
)

lgu_risk_ranking_boundary[
    ["Sensitivity_Rank", "LGU", "Structural_Risk_Score"]
].round(2)

,Sensitivity_Rank,LGU,Structural_Risk_Score
0,1,Pateros,264.40
1,2,Muntinlupa,253.15
2,3,Parañaque,251.38
3,4,Valenzuela,250.89
4,5,Pasig,247.90
5,6,Quezon City,247.31
6,7,Las Piñas,245.94
7,8,Marikina,241.65
8,9,San Juan,240.85
9,10,Malabon,237.53


In [32]:
# Makati–Taguig boundary-adjusted population sensitivity scenario
# IMPORTANT: This creates a COPY. The original panel is not modified.

panel_boundary_scenario = panel.copy()

# PSA retrospectively boundary-adjusted population anchors:
# Makati excludes the 10 transferred barangays.
# Taguig includes the 10 transferred barangays.

boundary_population_anchors = {
    "Makati": {
        2020: 292743,
        2024: 309770
    },
    "Taguig": {
        2020: 1223595,
        2024: 1308085
    }
}

# Reproduce the project's existing interpolation/extrapolation method,
# but use geographically comparable 2020 and 2024 anchors.

for lgu, anchors in boundary_population_anchors.items():

    annual_change = (
        anchors[2024] - anchors[2020]
    ) / 4

    for year in range(2021, 2026):

        scenario_population = (
            anchors[2020]
            + annual_change * (year - 2020)
        )

        mask = (
            (panel_boundary_scenario["LGU"] == lgu)
            & (panel_boundary_scenario["Year"] == year)
        )

        panel_boundary_scenario.loc[
            mask, "Population"
        ] = scenario_population


# Display original vs sensitivity populations

population_comparison = (
    panel[
        panel["LGU"].isin(["Makati", "Taguig"])
    ][["LGU", "Year", "Population"]]
    .rename(columns={"Population": "Current_Population"})
    .merge(
        panel_boundary_scenario[
            panel_boundary_scenario["LGU"].isin(
                ["Makati", "Taguig"]
            )
        ][["LGU", "Year", "Population"]]
        .rename(
            columns={
                "Population": "Boundary_Adjusted_Population"
            }
        ),
        on=["LGU", "Year"]
    )
)

population_comparison

,LGU,Year,Current_Population,Boundary_Adjusted_Population
0,Makati,2021,549654.50,296999.75
1,Makati,2022,469693.00,301256.50
2,Makati,2023,389731.50,305513.25
3,Makati,2024,309770.00,309770.00
4,Makati,2025,229808.50,314026.75
5,Taguig,2021,992062.75,1244717.50
6,Taguig,2022,1097403.50,1265840.00
7,Taguig,2023,1202744.25,1286962.50
8,Taguig,2024,1308085.00,1308085.00
9,Taguig,2025,1413425.75,1329207.50


In [33]:
# Recalculate population-dependent variables
# for the boundary-adjusted sensitivity scenario.

panel_boundary_scenario["Population Density"] = (
    panel_boundary_scenario["Population"]
    / panel_boundary_scenario["Land Area"]
)

panel_boundary_scenario["Incidence Rate"] = (
    panel_boundary_scenario["Dengue Cases"]
    / panel_boundary_scenario["Population"]
) * 100000


# Compare current vs boundary-adjusted values
# for Makati and Taguig only.

current_values = panel[
    panel["LGU"].isin(["Makati", "Taguig"])
][
    [
        "LGU",
        "Year",
        "Population Density",
        "Incidence Rate"
    ]
].copy()

current_values = current_values.rename(
    columns={
        "Population Density": "Current_Density",
        "Incidence Rate": "Current_Incidence"
    }
)

scenario_values = panel_boundary_scenario[
    panel_boundary_scenario["LGU"].isin(["Makati", "Taguig"])
][
    [
        "LGU",
        "Year",
        "Population Density",
        "Incidence Rate"
    ]
].copy()

scenario_values = scenario_values.rename(
    columns={
        "Population Density": "Scenario_Density",
        "Incidence Rate": "Scenario_Incidence"
    }
)

boundary_variable_comparison = current_values.merge(
    scenario_values,
    on=["LGU", "Year"]
)

boundary_variable_comparison.round(2)

,LGU,Year,Current_Density,Current_Incidence,Scenario_Density,Scenario_Incidence
0,Makati,2021,25482.36,83.14,13769.11,153.87
1,Makati,2022,21775.29,418.78,13966.46,652.93
2,Makati,2023,18068.22,291.48,14163.80,371.83
3,Makati,2024,14361.15,340.58,14361.15,340.58
4,Makati,2025,10654.08,479.96,14558.50,351.24
5,Taguig,2021,21943.44,129.93,27531.91,103.56
6,Taguig,2022,24273.47,467.01,27999.12,404.87
7,Taguig,2023,26603.50,222.99,28466.32,208.40
8,Taguig,2024,28933.53,209.24,28933.53,209.24
9,Taguig,2025,31263.56,206.59,29400.74,219.68


In [34]:
# Boundary-adjusted population scenario:
# Fit the same Negative Binomial model using all 17 LGUs / 85 observations.

panel_boundary_scenario["Density_per_1000"] = (
    panel_boundary_scenario["Population Density"] / 1000
)

year_dummies_scenario = pd.get_dummies(
    panel_boundary_scenario["Year"].astype(str),
    prefix="Year",
    drop_first=True
)

X_scenario = pd.concat(
    [
        panel_boundary_scenario[["Density_per_1000"]],
        year_dummies_scenario
    ],
    axis=1
)

X_scenario = sm.add_constant(X_scenario).astype(float)

y_scenario = panel_boundary_scenario[
    "Dengue Cases"
].astype(float)

offset_scenario = np.log(
    panel_boundary_scenario["Population"]
).astype(float)


# Fit the same NB2 model

nb_model_scenario = sm.NegativeBinomial(
    y_scenario,
    X_scenario,
    offset=offset_scenario,
    loglike_method="nb2"
)

nb_results_scenario = nb_model_scenario.fit(
    method="bfgs",
    maxiter=2000,
    disp=False,
    cov_type="cluster",
    cov_kwds={
        "groups": panel_boundary_scenario["LGU"]
    }
)

print(nb_results_scenario.summary())

                     NegativeBinomial Regression Results                      
Dep. Variable:           Dengue Cases   No. Observations:                   85
Model:               NegativeBinomial   Df Residuals:                       79
Method:                           MLE   Df Model:                            5
Date:                Fri, 25 Sep 2026   Pseudo R-squ.:                 0.06449
Time:                        14:48:02   Log-Likelihood:                -634.18
converged:                       True   LL-Null:                       -677.90
Covariance Type:              cluster   LLR p-value:                 2.317e-17
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -6.8738      0.157    -43.877      0.000      -7.181      -6.567
Density_per_1000    -0.0069      0.004     -1.574      0.115      -0.016       0.002
Year_2022            1.4202 

In [35]:
# Structural-risk ranking under the boundary-adjusted population scenario

beta_scenario = nb_results_scenario.params.drop("alpha")

mu_scenario = np.exp(
    np.dot(X_scenario, beta_scenario)
    + offset_scenario
)

structural_risk_scenario = panel_boundary_scenario[
    ["LGU", "Year", "Population", "Population Density"]
].copy()

structural_risk_scenario["Predicted_Cases"] = mu_scenario

structural_risk_scenario["Predicted_Incidence_per_100k"] = (
    structural_risk_scenario["Predicted_Cases"]
    / structural_risk_scenario["Population"]
) * 100000

lgu_risk_ranking_scenario = (
    structural_risk_scenario
    .groupby("LGU", as_index=False)
    .agg(
        Scenario_Structural_Risk_Score=(
            "Predicted_Incidence_per_100k",
            "mean"
        )
    )
    .sort_values(
        "Scenario_Structural_Risk_Score",
        ascending=False
    )
    .reset_index(drop=True)
)

lgu_risk_ranking_scenario["Scenario_Rank"] = (
    np.arange(1, len(lgu_risk_ranking_scenario) + 1)
)

# Compare against the current primary-model ranking

ranking_comparison = (
    lgu_risk_ranking[
        ["LGU", "Risk_Rank", "Structural_Risk_Score"]
    ]
    .merge(
        lgu_risk_ranking_scenario[
            [
                "LGU",
                "Scenario_Rank",
                "Scenario_Structural_Risk_Score"
            ]
        ],
        on="LGU"
    )
)

ranking_comparison["Rank_Change"] = (
    ranking_comparison["Risk_Rank"]
    - ranking_comparison["Scenario_Rank"]
)

ranking_comparison = ranking_comparison.sort_values(
    "Scenario_Rank"
).reset_index(drop=True)

ranking_comparison.round(2)

,LGU,Risk_Rank,Structural_Risk_Score,Scenario_Rank,Scenario_Structural_Risk_Score,Rank_Change
0,Pateros,1,273.33,1,277.20,0
1,Muntinlupa,2,260.38,2,263.33,0
2,Makati,5,255.31,3,262.66,2
3,Parañaque,3,258.35,4,261.15,-1
4,Valenzuela,4,257.78,5,260.55,-1
5,Pasig,6,254.36,6,256.90,0
6,Quezon City,7,253.68,7,256.17,0
7,Las Piñas,8,252.11,8,254.49,0
8,Marikina,9,247.21,9,249.27,0
9,San Juan,10,246.31,10,248.30,0


In [36]:
# Final Makati–Taguig sensitivity analysis:
# Compare Priority / Watch / Stable classifications
# under the primary model and boundary-adjusted scenario.

# Calculate five-year average observed incidence
scenario_incidence_summary = (
    panel_boundary_scenario
    .groupby("LGU", as_index=False)
    .agg(
        Scenario_Five_Year_Avg_Incidence=(
            "Incidence Rate",
            "mean"
        )
    )
)

# Get 2025 observed incidence
scenario_2025 = (
    panel_boundary_scenario[
        panel_boundary_scenario["Year"] == 2025
    ][["LGU", "Incidence Rate"]]
    .rename(
        columns={
            "Incidence Rate": "Scenario_Incidence_2025"
        }
    )
)

# Combine scenario ranking + incidence information
scenario_priority = (
    lgu_risk_ranking_scenario
    .merge(
        scenario_incidence_summary,
        on="LGU"
    )
    .merge(
        scenario_2025,
        on="LGU"
    )
)

# Same top-third rule used by the project.
# With 17 LGUs, ranks 1–6 are the top third.
scenario_priority["Scenario_Top_Third_Risk"] = (
    scenario_priority["Scenario_Rank"] <= 6
)

scenario_priority["Scenario_2025_At_or_Above_5Yr_Avg"] = (
    scenario_priority["Scenario_Incidence_2025"]
    >= scenario_priority["Scenario_Five_Year_Avg_Incidence"]
)

# Apply the exact same Priority / Watch / Stable logic.
scenario_priority["Scenario_Priority_Tier"] = (
    scenario_priority.apply(
        lambda row: assign_priority_tier(
            row["Scenario_Top_Third_Risk"],
            row["Scenario_2025_At_or_Above_5Yr_Avg"]
        ),
        axis=1
    )
)

# Compare Makati and Taguig only.
scenario_priority[
    scenario_priority["LGU"].isin(["Makati", "Taguig"])
][
    [
        "LGU",
        "Scenario_Rank",
        "Scenario_Structural_Risk_Score",
        "Scenario_Incidence_2025",
        "Scenario_Five_Year_Avg_Incidence",
        "Scenario_Top_Third_Risk",
        "Scenario_2025_At_or_Above_5Yr_Avg",
        "Scenario_Priority_Tier"
    ]
].round(2)

,LGU,Scenario_Rank,Scenario_Structural_Risk_Score,Scenario_Incidence_2025,Scenario_Five_Year_Avg_Incidence,Scenario_Top_Third_Risk,Scenario_2025_At_or_Above_5Yr_Avg,Scenario_Priority_Tier
2,Makati,3,262.66,351.24,374.09,True,False,Watch
12,Taguig,13,237.76,219.68,229.15,False,False,Stable


In [20]:
# Random Forest robustness comparison using grouped cross-validation by LGU

rf_model = RandomForestRegressor(
    n_estimators=500,
    random_state=42,
    min_samples_leaf=2
)

# 5-fold grouped cross-validation
group_cv = GroupKFold(n_splits=5)

rf_predictions = cross_val_predict(
    rf_model,
    X_rf,
    y_rf,
    cv=group_cv,
    groups=groups
)

rf_mae = mean_absolute_error(y_rf, rf_predictions)
rf_rmse = np.sqrt(mean_squared_error(y_rf, rf_predictions))
rf_r2 = r2_score(y_rf, rf_predictions)

print(f"Random Forest Group-CV MAE: {rf_mae:.2f}")
print(f"Random Forest Group-CV RMSE: {rf_rmse:.2f}")
print(f"Random Forest Group-CV R²: {rf_r2:.4f}")

Random Forest Group-CV MAE: 758.20
Random Forest Group-CV RMSE: 1412.80
Random Forest Group-CV R²: 0.5536


In [21]:
# Compare Negative Binomial predictions against Random Forest

nb_predictions = mu_hat

nb_mae = mean_absolute_error(y_nb, nb_predictions)
nb_rmse = np.sqrt(mean_squared_error(y_nb, nb_predictions))
nb_r2 = r2_score(y_nb, nb_predictions)

comparison_table = pd.DataFrame({
    "Model": [
        "Negative Binomial",
        "Random Forest"
    ],
    "MAE": [
        nb_mae,
        rf_mae
    ],
    "RMSE": [
        nb_rmse,
        rf_rmse
    ],
    "R2": [
        nb_r2,
        rf_r2
    ]
})

comparison_table.round({
    "MAE": 2,
    "RMSE": 2,
    "R2": 4
})

,Model,MAE,RMSE,R2
0,Negative Binomial,498.9,822.62,0.8486
1,Random Forest,758.2,1412.80,0.5536


In [22]:
import warnings

# Fair grouped cross-validation comparison for Negative Binomial
# Uses the same 5-fold GroupKFold structure as Random Forest.

nb_cv_predictions = np.full(len(panel), np.nan)
nb_cv_fold_results = []

group_cv = GroupKFold(n_splits=5)

# Use full-model estimates as starting values
full_start_params = nb_results.params.values

for fold_number, (train_idx, test_idx) in enumerate(
    group_cv.split(
        panel,
        panel["Dengue Cases"],
        groups=panel["LGU"]
    ),
    start=1
):

    train = panel.iloc[train_idx].copy()
    test = panel.iloc[test_idx].copy()

    train["Density_per_1000"] = (
        train["Population Density"] / 1000
    )

    test["Density_per_1000"] = (
        test["Population Density"] / 1000
    )

    # Year dummy variables
    train_year_dummies = pd.get_dummies(
        train["Year"].astype(str),
        prefix="Year",
        drop_first=True
    )

    test_year_dummies = pd.get_dummies(
        test["Year"].astype(str),
        prefix="Year",
        drop_first=True
    )

    test_year_dummies = test_year_dummies.reindex(
        columns=train_year_dummies.columns,
        fill_value=0
    )

    X_train_nb = pd.concat(
        [
            train[["Density_per_1000"]],
            train_year_dummies
        ],
        axis=1
    )

    X_test_nb = pd.concat(
        [
            test[["Density_per_1000"]],
            test_year_dummies
        ],
        axis=1
    )

    X_train_nb = sm.add_constant(
        X_train_nb,
        has_constant="add"
    ).astype(float)

    X_test_nb = sm.add_constant(
        X_test_nb,
        has_constant="add"
    ).astype(float)

    X_test_nb = X_test_nb.reindex(
        columns=X_train_nb.columns,
        fill_value=0
    )

    y_train_nb = train["Dengue Cases"].astype(float)

    offset_train_nb = np.log(
        train["Population"]
    ).astype(float)

    offset_test_nb = np.log(
        test["Population"]
    ).astype(float)

    nb_cv_model = sm.NegativeBinomial(
        y_train_nb,
        X_train_nb,
        offset=offset_train_nb,
        loglike_method="nb2"
    )

    # Suppress temporary numerical warnings,
    # but record final convergence explicitly.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)

        nb_cv_fit = nb_cv_model.fit(
            start_params=full_start_params,
            method="bfgs",
            maxiter=3000,
            disp=False
        )

    converged = nb_cv_fit.mle_retvals["converged"]

    beta_cv = nb_cv_fit.params.drop("alpha")

    nb_cv_predictions[test_idx] = np.exp(
        np.dot(X_test_nb, beta_cv)
        + offset_test_nb
    )

    nb_cv_fold_results.append({
        "Fold": fold_number,
        "Converged": converged,
        "Alpha": nb_cv_fit.params["alpha"],
        "Log_Likelihood": nb_cv_fit.llf
    })


# Check fold convergence
nb_cv_fold_table = pd.DataFrame(nb_cv_fold_results)

print("Fold convergence:")
display(nb_cv_fold_table.round({
    "Alpha": 4,
    "Log_Likelihood": 2
}))

# Compute metrics only after confirming predictions exist
nb_cv_mae = mean_absolute_error(
    y_nb,
    nb_cv_predictions
)

nb_cv_rmse = np.sqrt(
    mean_squared_error(
        y_nb,
        nb_cv_predictions
    )
)

nb_cv_r2 = r2_score(
    y_nb,
    nb_cv_predictions
)

print(f"\nNegative Binomial Group-CV MAE: {nb_cv_mae:.2f}")
print(f"Negative Binomial Group-CV RMSE: {nb_cv_rmse:.2f}")
print(f"Negative Binomial Group-CV R²: {nb_cv_r2:.4f}")

Fold convergence:


,Fold,Converged,Alpha,Log_Likelihood
0,1,True,0.1354,-492.90
1,2,True,0.0804,-464.29
2,3,True,0.1426,-523.29
3,4,True,0.1081,-528.65
4,5,True,0.1221,-507.38



Negative Binomial Group-CV MAE: 585.51
Negative Binomial Group-CV RMSE: 1045.09
Negative Binomial Group-CV R²: 0.7557


In [23]:
# Task 10 — Negative Binomial structural risk ranking
# Rank LGUs using model-adjusted predicted incidence per 100,000 population.

structural_risk = panel[
    ["LGU", "Year", "Population", "Population Density"]
].copy()

# Full-model predicted dengue case counts
structural_risk["Predicted_Cases"] = mu_hat

# Convert fitted counts to predicted incidence per 100,000
structural_risk["Predicted_Incidence_per_100k"] = (
    structural_risk["Predicted_Cases"]
    / structural_risk["Population"]
) * 100000

# Average model-adjusted incidence across 2021–2025 for each LGU
lgu_risk_ranking = (
    structural_risk
    .groupby("LGU", as_index=False)
    .agg(
        Structural_Risk_Score=(
            "Predicted_Incidence_per_100k",
            "mean"
        ),
        Mean_Population_Density=(
            "Population Density",
            "mean"
        )
    )
)

# Rank from highest modeled risk to lowest
lgu_risk_ranking = lgu_risk_ranking.sort_values(
    "Structural_Risk_Score",
    ascending=False
).reset_index(drop=True)

lgu_risk_ranking["Risk_Rank"] = (
    np.arange(1, len(lgu_risk_ranking) + 1)
)

# Reorder columns
lgu_risk_ranking = lgu_risk_ranking[
    [
        "Risk_Rank",
        "LGU",
        "Structural_Risk_Score",
        "Mean_Population_Density"
    ]
]

lgu_risk_ranking.round({
    "Structural_Risk_Score": 2,
    "Mean_Population_Density": 2
})

,Risk_Rank,LGU,Structural_Risk_Score,Mean_Population_Density
0,1,Pateros,273.33,6422.69
1,2,Muntinlupa,260.38,13837.23
2,3,Parañaque,258.35,15029.67
3,4,Valenzuela,257.78,15368.44
4,5,Makati,255.31,18068.22
5,6,Pasig,254.36,17345.80
6,7,Quezon City,253.68,17781.23
7,8,Las Piñas,252.11,18759.10
8,9,Marikina,247.21,21724.30
9,10,San Juan,246.31,22238.78


In [24]:
# Task 11 — Apply Priority / Watch / Stable rule

# Compute observed five-year average incidence and 2025 incidence by LGU
observed_indicators = (
    panel.groupby("LGU", as_index=False)
    .agg(
        Five_Year_Avg_Incidence=("Incidence Rate", "mean")
    )
)

incidence_2025 = (
    panel.loc[
        panel["Year"] == 2025,
        ["LGU", "Incidence Rate"]
    ]
    .rename(
        columns={
            "Incidence Rate": "Incidence_2025"
        }
    )
)

# Merge with structural risk ranking
priority_flags = (
    lgu_risk_ranking
    .merge(
        observed_indicators,
        on="LGU",
        how="left"
    )
    .merge(
        incidence_2025,
        on="LGU",
        how="left"
    )
)

# Top third of 17 LGUs = ranks 1 to 6
top_third_cutoff = int(
    np.ceil(len(priority_flags) / 3)
)

priority_flags["Top_Third_Risk"] = (
    priority_flags["Risk_Rank"] <= top_third_cutoff
)

priority_flags["2025_At_or_Above_5Yr_Avg"] = (
    priority_flags["Incidence_2025"]
    >= priority_flags["Five_Year_Avg_Incidence"]
)

# Apply tier rule
def assign_priority_tier(row):
    if (
        row["Top_Third_Risk"]
        and row["2025_At_or_Above_5Yr_Avg"]
    ):
        return "Priority"

    elif (
        row["Top_Third_Risk"]
        or row["2025_At_or_Above_5Yr_Avg"]
    ):
        return "Watch"

    else:
        return "Stable"


priority_flags["Priority_Tier"] = (
    priority_flags.apply(
        assign_priority_tier,
        axis=1
    )
)

priority_flags[
    [
        "Risk_Rank",
        "LGU",
        "Structural_Risk_Score",
        "Incidence_2025",
        "Five_Year_Avg_Incidence",
        "Top_Third_Risk",
        "2025_At_or_Above_5Yr_Avg",
        "Priority_Tier"
    ]
].round({
    "Structural_Risk_Score": 2,
    "Incidence_2025": 2,
    "Five_Year_Avg_Incidence": 2
})

,Risk_Rank,LGU,Structural_Risk_Score,Incidence_2025,Five_Year_Avg_Incidence,Top_Third_Risk,2025_At_or_Above_5Yr_Avg,Priority_Tier
0,1,Pateros,273.33,347.87,438.65,True,False,Watch
1,2,Muntinlupa,260.38,176.76,138.03,True,True,Priority
2,3,Parañaque,258.35,228.01,209.38,True,True,Priority
3,4,Valenzuela,257.78,246.66,300.87,True,False,Watch
4,5,Makati,255.31,479.96,322.79,True,True,Priority
5,6,Pasig,254.36,421.02,299.88,True,True,Priority
6,7,Quezon City,253.68,355.37,229.73,False,True,Watch
7,8,Las Piñas,252.11,345.06,218.29,False,True,Watch
8,9,Marikina,247.21,245.40,211.15,False,True,Watch
9,10,San Juan,246.31,228.90,212.96,False,True,Watch


In [25]:
def assign_priority_tier(top_third_risk, incidence_at_or_above_avg):
    """
    Assign Priority / Watch / Stable classification.

    Priority:
        Top-third structural risk AND
        2025 incidence >= five-year average.

    Watch:
        Exactly one of the two conditions is true.

    Stable:
        Neither condition is true.
    """

    if top_third_risk and incidence_at_or_above_avg:
        return "Priority"

    if top_third_risk or incidence_at_or_above_avg:
        return "Watch"

    return "Stable"

In [26]:
from src.priority import assign_priority_tier

hand_checks = pd.DataFrame([
    {
        "LGU": "Muntinlupa",
        "Top_Third_Risk": True,
        "2025_At_or_Above_5Yr_Avg": True,
        "Expected": "Priority"
    },
    {
        "LGU": "Pateros",
        "Top_Third_Risk": True,
        "2025_At_or_Above_5Yr_Avg": False,
        "Expected": "Watch"
    },
    {
        "LGU": "Manila",
        "Top_Third_Risk": False,
        "2025_At_or_Above_5Yr_Avg": True,
        "Expected": "Watch"
    },
    {
        "LGU": "Taguig",
        "Top_Third_Risk": False,
        "2025_At_or_Above_5Yr_Avg": False,
        "Expected": "Stable"
    }
])

hand_checks["Actual"] = hand_checks.apply(
    lambda row: assign_priority_tier(
        row["Top_Third_Risk"],
        row["2025_At_or_Above_5Yr_Avg"]
    ),
    axis=1
)

hand_checks["Match"] = (
    hand_checks["Expected"] == hand_checks["Actual"]
)

hand_checks

,LGU,Top_Third_Risk,2025_At_or_Above_5Yr_Avg,Expected,Actual,Match
0,Muntinlupa,True,True,Priority,Priority,True
1,Pateros,True,False,Watch,Watch,True
2,Manila,False,True,Watch,Watch,True
3,Taguig,False,False,Stable,Stable,True


In [27]:
# Task 14 — Export dashboard-ready CSV files

output_dir = Path("../outputs")
output_dir.mkdir(exist_ok=True)

# 1. LGU structural risk ranking
risk_export = lgu_risk_ranking.copy()

risk_export.to_csv(
    output_dir / "lgu_structural_risk_ranking.csv",
    index=False
)


# 2. Priority / Watch / Stable flags
priority_export = priority_flags[
    [
        "Risk_Rank",
        "LGU",
        "Structural_Risk_Score",
        "Incidence_2025",
        "Five_Year_Avg_Incidence",
        "Top_Third_Risk",
        "2025_At_or_Above_5Yr_Avg",
        "Priority_Tier"
    ]
].copy()

priority_export.to_csv(
    output_dir / "lgu_priority_flags.csv",
    index=False
)


# 3. LGU-year model output for dashboard trends
model_output_export = panel[
    [
        "LGU",
        "Year",
        "Dengue Cases",
        "Population",
        "Population Density",
        "Incidence Rate"
    ]
].copy()

model_output_export["Predicted_Cases_NB"] = mu_hat

model_output_export["Predicted_Incidence_per_100k"] = (
    model_output_export["Predicted_Cases_NB"]
    / model_output_export["Population"]
) * 100000

model_output_export.to_csv(
    output_dir / "lgu_year_nb_model_output.csv",
    index=False
)


print("Exported files:")
print("1. lgu_structural_risk_ranking.csv")
print("2. lgu_priority_flags.csv")
print("3. lgu_year_nb_model_output.csv")

Exported files:
1. lgu_structural_risk_ranking.csv
2. lgu_priority_flags.csv
3. lgu_year_nb_model_output.csv


### Makati–Taguig Boundary Sensitivity Analysis

Due to the administrative transfer of 10 barangays from Makati to Taguig creating a geographic discontinuity between the population reference years used in the study, two sensitivity tests were conducted.

First, Makati and Taguig were excluded from the Negative Binomial model. The density coefficient remained negative and statistically non-significant, changing from -0.0065 (p = 0.1296) in the primary model to -0.0059 (p = 0.1843). The relative structural-risk ordering of the remaining 15 LGUs was unchanged.

Second, a boundary-adjusted population scenario was evaluated using retrospectively comparable 2020 and 2024 population anchors for Makati and Taguig while retaining the study's linear interpolation/extrapolation procedure. Under this scenario, the density coefficient remained negative and statistically non-significant at -0.0069 (p = 0.115), and the year effects and dispersion parameter remained similar to the primary model.

The sensitivity analyses therefore indicate that the Makati–Taguig boundary issue does not materially alter the overall interpretation of the Negative Binomial model or the relative structural-risk ordering of the unaffected LGUs.

However, the downstream classification of Makati was sensitive to the alternative population treatment. Makati changed from rank 5 (Priority) in the primary analysis to rank 3 (Watch) in the boundary-adjusted scenario because its scenario 2025 incidence (351.24 per 100,000) was below its scenario five-year average incidence (374.09 per 100,000). Taguig changed from rank 12 to rank 13 but remained classified as Stable.

The boundary-adjusted values are treated as a sensitivity scenario rather than replacements for the primary analytical dataset because barangay-level dengue case counts were not available to verify that historical dengue numerators could be harmonized to the same retrospective geographic boundaries used for the alternative population denominators.